# Pancancer enrichment analysis step 7: Make figures from GSEApy data

## Setup

In [1]:
import os
import datetime
import altair as alt
import numpy as np
import pandas as pd
import operator
import IPython.display

In [2]:
TIME_START = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

# Step 1 output
STEP1_DIR = "step1_outputs"
STEP1_FILE = "tumor_change_20200529_195104_10000_perm.tsv"
STEP1_FILE_PATH = os.path.join(STEP1_DIR, STEP1_FILE)

# Step 5 outputs
STEP5_DIR = "step5_outputs"

STEP5_abs_signal_to_noise = os.path.join(STEP5_DIR, "enrichment_gseapy_reactome_20200617_192305_10000_perms_abs_signal_to_noise.tsv")
STEP5_diff_of_classes = os.path.join(STEP5_DIR, "enrichment_gseapy_reactome_20200617_192305_10000_perms_diff_of_classes.tsv")
STEP5_log2_ratio_of_classes = os.path.join(STEP5_DIR, "enrichment_gseapy_reactome_20200617_192305_10000_perms_log2_ratio_of_classes.tsv")
STEP5_ratio_of_classes = os.path.join(STEP5_DIR, "enrichment_gseapy_reactome_20200617_192305_10000_perms_ratio_of_classes.tsv")
STEP5_signal_to_noise = os.path.join(STEP5_DIR, "enrichment_gseapy_reactome_20200617_192305_10000_perms_signal_to_noise.tsv")
STEP5_t_test = os.path.join(STEP5_DIR, "enrichment_gseapy_reactome_20200617_192305_10000_perms_t_test.tsv")

# Step 6 output
STEP6_DIR = "step6_outputs"

STEP6_S2N = "enrichment_gseapy_kegg_signal_to_noise_10000_perms_20200617_072701.tsv"
STEP6_S2N_PATH = os.path.join(STEP6_DIR, STEP6_S2N)

STEP6_ABS_S2N = "enrichment_gseapy_kegg_abs_signal_to_noise_10000_perms_20200617_103905.tsv"
STEP6_ABS_S2N_PATH = os.path.join(STEP6_DIR, STEP6_ABS_S2N)

MSV = 0.5 # Simplified expression value for marginally significant proteins

In [3]:
# Altair options
# alt.data_transformers.disable_max_rows()
# alt.data_transformers.enable('json')

## Function to plot top pathways across cancer types

In [4]:
def plot_top_ten(enrich_file_path, expr_file_path, xtitle, fdr=0.3):
    
    # Read in the expression data
    all_expression_data = pd.read_csv(expr_file_path, sep="\t", index_col=0)

    # Make a column where all increases are +1 and all decreases 
    # are -1, because these are ratioed abundances, so we can't 
    # compare magnitudes between genes--we can only compare whether 
    # there was a change, and whether it was positive or negative
    all_expression_data = all_expression_data.assign(simplified_change=np.nan)

    # adj p < 0.05 and change > 1 => +1
    all_expression_data.loc[
        (all_expression_data["change"] > 0) & (all_expression_data["adj_p"] < 0.05), 
        "simplified_change"
    ] = 1

    # adj p >= 0.05 and change > 1 => +0.5
    all_expression_data.loc[(all_expression_data["change"] > 0) & (all_expression_data["adj_p"] >= 0.05),
        "simplified_change"
    ] = MSV

    # change == 0 => 0
    all_expression_data.loc[
        all_expression_data["change"] == 0,
        "simplified_change"
    ] = 0

    # adj p >= 0.05 and change < 1 => -0.5
    all_expression_data.loc[(all_expression_data["change"] < 0) & (all_expression_data["adj_p"] >= 0.05), 
        "simplified_change"
    ] = -MSV

    # adj p < 0.05 and change < 1 => -1
    all_expression_data.loc[
        (all_expression_data["change"] < 0) & (all_expression_data["adj_p"] < 0.05),
        "simplified_change"
    ] = -1

    # Select just the proteins where we chose to reject the null hypothesis of no change
    expression_data = all_expression_data[all_expression_data["reject_null"]]

    # Read in the enrichment data
    enrichment_data = pd.read_csv(enrich_file_path, sep="\t")
    
    # Select enrichment data below our FDR
    enrichment_data = enrichment_data.loc[enrichment_data["fdr"] <= fdr, :]
    
    # Make a column for abs_nes
    enrichment_data = enrichment_data.assign(abs_nes=enrichment_data["nes"].abs())

    # Assign pathway ranks within each cancer type based on absolute value of NES
    enrichment_data = enrichment_data.\
        assign(cancer_rank_abs_nes=enrichment_data.groupby("cancer_type")["abs_nes"].rank(ascending=False)).\
        sort_values(by=["cancer_type", "cancer_rank_abs_nes"]).\
        reset_index(drop=True)

    # Make a table with summary info for all pathways
    enrichment_summary = enrichment_data[["Term"]].drop_duplicates(keep="first")

    pathway_times_enriched = enrichment_summary["Term"].apply(
        lambda x: enrichment_data[enrichment_data["Term"] == x].shape[0])

    avg_rank = enrichment_summary["Term"].apply(
        lambda x: enrichment_data.loc[enrichment_data["Term"] == x, "cancer_rank_abs_nes"].mean())

    enrichment_summary = enrichment_summary.\
        assign(
            pathway_times_enriched=pathway_times_enriched,
            pathway_avg_rank=avg_rank).\
        sort_values(
            by=["pathway_times_enriched", "pathway_avg_rank"],
            ascending=[False, True]).\
        reset_index(drop=True)

    # Merge in the original enrichment data
    enrichment_data = enrichment_data.\
        merge(
            enrichment_summary,
            how="outer",
            left_on="Term",
            right_on="Term",
            validate="many_to_one"
        ).\
        sort_values(
            by=["pathway_times_enriched", "pathway_avg_rank", "cancer_type"],
            ascending=[False, True, True]
        )

    # Select top 10 for our plot
    top_ten = enrichment_data["Term"].unique()[:10]
    sel_enrich = enrichment_data[enrichment_data["Term"].isin(top_ten)]

    # Calculate the mean expression for each pathway in each cancer type
    mean_exprs = []

    for idx in sel_enrich.index:
        genes = sel_enrich.loc[idx, "genes"].split(";")
        cancer_type = sel_enrich.loc[idx, "cancer_type"]

        genes_expr = expression_data.loc[
            expression_data["protein_str"].isin(genes),
            "simplified_change"
        ].mean()

        mean_exprs.append(genes_expr)

    sel_enrich = sel_enrich.assign(mean_expr=mean_exprs)

    sel_enrich = sel_enrich.assign(
        rank_size=1 / sel_enrich["cancer_rank_abs_nes"],
        avg_rank_size=1 / sel_enrich["pathway_avg_rank"])

    individual = alt.Chart(sel_enrich).mark_circle().encode(
        x=alt.X(
            "Term:N",
            sort=sel_enrich["Term"].values,
            axis=alt.Axis(
                labelAngle=-30,
                labelFontSize=12,
                labelLimit=500,
                title="",
                titleFontSize=16
            )
        ),
        y=alt.Y(
            "cancer_type:N",
            axis=alt.Axis(
                title="Cancer type",
                titleFontSize=12
            ),
        ),
        size=alt.Size(
            "rank_size:Q",
            legend=None
        ),
        color=alt.Color(
            "mean_expr:Q",
            scale=alt.Scale(
                scheme="blueorange",
                domain=[-1, 1]
            ),
            legend=alt.Legend(
                title="Pathway tumor expression"
            )
        )
    ).properties(
        width=400,
        height=300
    )

    aggregate = alt.Chart(sel_enrich).mark_circle().encode(
        x=alt.X(
            "pathway_avg_rank:N",
            sort=sel_enrich["pathway_avg_rank"].values,
            axis=alt.Axis(
                labelAngle=-30,
                labelFontSize=12,
                labelLimit=500,
                title="Overall rank of pathway",
                titleFontSize=12
            )
        ),
        size=alt.Size(
            "avg_rank_size:Q",
            legend=None
        ),
    ).properties(
        width=400
    )

    full_plot = alt.vconcat(
        aggregate, individual
    ).properties(
        title=xtitle
    )
    
    return full_plot, enrichment_data, all_expression_data, enrichment_summary

In [5]:
plot_top_ten(STEP5_abs_signal_to_noise, STEP1_FILE_PATH, "Reactome data - proteins ranked by abs_signal_to_noise")[3]

,Term,pathway_times_enriched,pathway_avg_rank
0,Terminal pathway of complement,3,2.333333
1,Erythrocytes take up oxygen and release carbon...,2,1.000000
2,Defective factor XII causes hereditary angioedema,2,5.000000
3,Amine Oxidase reactions,2,8.000000
4,Biogenic amines are oxidatively deaminated to ...,2,8.000000
5,Defective MAOA causes Brunner syndrome (BRUNS),2,14.500000
6,Glyoxylate metabolism and glycine degradation,2,45.500000
7,RNA Polymerase II Pre-transcription Events,1,1.000000
8,Retrograde transport at the Trans-Golgi-Network,1,1.000000
9,Class C/3 (Metabotropic glutamate/pheromone re...,1,1.000000


In [6]:
plot_top_ten(STEP5_diff_of_classes, STEP1_FILE_PATH, "Reactome data - proteins ranked by diff_of_classes")[3]

,Term,pathway_times_enriched,pathway_avg_rank
0,Major pathway of rRNA processing in the nucleo...,5,35.800
1,Activation of the mRNA upon binding of the cap...,5,66.400
2,Ribosomal scanning and start codon recognition,5,70.800
3,Translation initiation complex formation,5,73.400
4,Transport of Mature mRNA Derived from an Intro...,4,13.250
5,NS1 Mediated Effects on Host Pathways,4,17.250
6,mRNA Splicing - Major Pathway,4,19.500
7,Metabolism of non-coding RNA,4,23.500
8,snRNP Assembly,4,23.500
9,Transport of the SLBP Dependant Mature mRNA,4,31.250


In [7]:
plot_top_ten(STEP5_log2_ratio_of_classes, STEP1_FILE_PATH, "Reactome data - proteins ranked by log2_ratio_of_classes")[3]

,Term,pathway_times_enriched,pathway_avg_rank


In [8]:
plot_top_ten(STEP5_ratio_of_classes, STEP1_FILE_PATH, "Reactome data - proteins ranked by ratio_of_classes")[3]

,Term,pathway_times_enriched,pathway_avg_rank
0,Toxicity of botulinum toxin type D (BoNT/D),2,28.75
1,Toxicity of botulinum toxin type F (BoNT/F),2,28.75
2,Synthesis of 12-eicosatetraenoic acid derivatives,2,29.50
3,Myoclonic epilepsy of Lafora,2,36.50
4,Passive transport by Aquaporins,2,36.50
5,CREB phosphorylation,2,46.50
6,Hypusine synthesis from eIF5A-lysine,2,53.50
7,IKBKB deficiency causes SCID,2,66.00
8,IKBKG deficiency causes anhidrotic ectodermal ...,2,66.00
9,Potassium Channels,2,75.00


In [9]:
plot_top_ten(STEP5_signal_to_noise, STEP1_FILE_PATH, "Reactome data - proteins ranked by signal_to_noise")[3]

,Term,pathway_times_enriched,pathway_avg_rank
0,Paradoxical activation of RAF signaling by kin...,8,160.625000
1,Signaling by RAS mutants,8,160.625000
2,Signaling by moderate kinase activity BRAF mut...,8,160.625000
3,Signaling downstream of RAS mutants,8,160.625000
4,Signaling by RAF1 mutants,8,192.875000
5,NCAM signaling for neurite out-growth,7,70.428571
6,G alpha (q) signalling events,7,74.857143
7,Platelet homeostasis,7,109.142857
8,"PI5P, PP2A and IER3 Regulate PI3K/AKT Signaling",7,129.428571
9,GPCR ligand binding,7,137.000000


In [10]:
plot_top_ten(STEP5_t_test, STEP1_FILE_PATH, "Reactome data - proteins ranked by t_test")[3]

,Term,pathway_times_enriched,pathway_avg_rank
0,Paradoxical activation of RAF signaling by kin...,8,158.625000
1,Signaling by RAS mutants,8,158.625000
2,Signaling by moderate kinase activity BRAF mut...,8,158.625000
3,Signaling downstream of RAS mutants,8,158.625000
4,Signaling by RAF1 mutants,8,194.625000
5,Signaling by BRAF and RAF fusions,8,288.500000
6,NCAM signaling for neurite out-growth,7,72.285714
7,G alpha (q) signalling events,7,82.142857
8,Platelet homeostasis,7,111.285714
9,"PI5P, PP2A and IER3 Regulate PI3K/AKT Signaling",7,122.571429


In [11]:
df = pd.read_csv(STEP5_log2_ratio_of_classes, sep="\t", index_col=0)

In [12]:
df["fdr"].min()

0.8541666825504897

In [13]:
alt.vconcat(
    plot_top_ten(STEP5_abs_signal_to_noise, STEP1_FILE_PATH, "Reactome data - proteins ranked by abs_signal_to_noise")[0],
    plot_top_ten(STEP5_diff_of_classes, STEP1_FILE_PATH, "Reactome data - proteins ranked by diff_of_classes")[0],
    plot_top_ten(STEP5_log2_ratio_of_classes, STEP1_FILE_PATH, "Reactome data - proteins ranked by log2_ratio_of_classes")[0],
    plot_top_ten(STEP5_ratio_of_classes, STEP1_FILE_PATH, "Reactome data - proteins ranked by ratio_of_classes")[0],
    plot_top_ten(STEP5_signal_to_noise, STEP1_FILE_PATH, "Reactome data - proteins ranked by signal_to_noise")[0],
    plot_top_ten(STEP5_t_test, STEP1_FILE_PATH, "Reactome data - proteins ranked by t_test")[0],
    
    plot_top_ten(STEP6_ABS_S2N_PATH, STEP1_FILE_PATH, "KEGG data - proteins ranked by |S2N|")[0],
    plot_top_ten(STEP6_S2N_PATH, STEP1_FILE_PATH, "KEGG data - proteins ranked by S2N")[0],
).configure_axis(
    grid=True
).configure_title(
    fontSize=16,
    anchor="end",
    offset=20
)

alt.VConcatChart(...)

In [27]:
def find_neutrophil_degranulation(fp):
    enrich = plot_top_ten(fp, STEP1_FILE_PATH, "Reactome data - proteins ranked by |S2N|", fdr=1)[1]

    # enrich = enrich[enrich["pathway_times_enriched"] == 8]
    ranks = enrich[["Term", "pathway_avg_rank"]].\
        set_index("Term").\
        drop_duplicates(keep="first").\
        rank()["pathway_avg_rank"].\
        rename("pathway_overall_rank")

    enrich = enrich.merge(
        right=ranks,
        left_on="Term",
        right_on="Term",
        validate="many_to_one")

    return enrich[enrich["Term"] == "Neutrophil degranulation"]

In [28]:
find_neutrophil_degranulation(STEP5_abs_signal_to_noise)

,Term,es,nes,pval,fdr,geneset_size,matched_size,genes,ledge_genes,cancer_type,abs_nes,cancer_rank_abs_nes,pathway_times_enriched,pathway_avg_rank,pathway_overall_rank
10408,Neutrophil degranulation,0.443041,1.474740,0.000000,0.548928,479,427,PYGL;GGH;NDUFC2;TYROBP;FCER1G;ITGAX;RAP2B;LPCA...,PYGL;GGH;NDUFC2;TYROBP;FCER1G;ITGAX;RAP2B;LPCA...,ccrcc,1.474740,922.0,8,1426.75,1483.0
10409,Neutrophil degranulation,0.337260,1.095910,0.228904,0.703070,479,405,METTL7A;S100A11;HSP90AB1;AGL;CNN2;RAB5B;CFD;XR...,METTL7A;S100A11;HSP90AB1;AGL;CNN2;RAB5B;CFD;XR...,colon,1.095910,1582.0,8,1426.75,1483.0
10410,Neutrophil degranulation,0.350014,1.141544,0.176256,0.668910,479,427,AHSG;EEF2;SERPINA1;TIMP2;COPB1;A1BG;PSMD2;DYNL...,AHSG;EEF2;SERPINA1;TIMP2;COPB1;A1BG;PSMD2;DYNL...,endometrial,1.141544,1474.0,8,1426.75,1483.0
10411,Neutrophil degranulation,0.334066,1.095346,0.297285,0.752959,479,432,AP2A2;HSPA8;TOLLIP;ATP6V1D;LRRC7;NDUFC2;KCNAB2...,AP2A2;HSPA8;TOLLIP;ATP6V1D;LRRC7;NDUFC2;KCNAB2...,gbm,1.095346,1589.0,8,1426.75,1483.0
10412,Neutrophil degranulation,0.460514,1.474212,0.002217,0.364215,479,448,SPTAN1;AHSG;TTR;CFD;ORM2;A1BG;SERPINA1;CNN2;RA...,SPTAN1;AHSG;TTR;CFD;ORM2;A1BG;SERPINA1;CNN2;RA...,hnscc,1.474212,1096.0,8,1426.75,1483.0
10413,Neutrophil degranulation,0.309145,1.009420,0.417937,0.819957,479,423,SPTAN1;EEF2;VAT1;CD93;CD55;CAT;SERPINB6;PECAM1...,SPTAN1;EEF2;VAT1;CD93;CD55;CAT;SERPINB6;PECAM1...,lscc,1.009420,1734.0,8,1426.75,1483.0
10414,Neutrophil degranulation,0.311450,1.003927,0.465463,0.798988,479,409,SPTAN1;CAT;PECAM1;VCL;ANXA2;CD36;HBB;CD93;RAP1...,SPTAN1;CAT;PECAM1;VCL;ANXA2;CD36;HBB;CD93;RAP1...,luad,1.003927,1778.0,8,1426.75,1483.0
10415,Neutrophil degranulation,0.384449,1.233823,0.034091,0.676408,479,418,HSP90AB1;STOM;PRSS3;HSP90AA1;ILF2;COPB1;PECAM1...,HSP90AB1;STOM;PRSS3;HSP90AA1;ILF2;COPB1;PECAM1...,ovarian,1.233823,1239.0,8,1426.75,1483.0


In [29]:
find_neutrophil_degranulation(STEP5_diff_of_classes)

,Term,es,nes,pval,fdr,geneset_size,matched_size,genes,ledge_genes,cancer_type,abs_nes,cancer_rank_abs_nes,pathway_times_enriched,pathway_avg_rank,pathway_overall_rank
8184,Neutrophil degranulation,0.402375,2.329713,0.228597,0.484294,479,427,FCER1G;TYROBP;FTL;PYGL;FTH1;ITGAX;SLC2A3;CYBB;...,FCER1G;TYROBP;FTL;PYGL;FTH1;ITGAX;SLC2A3;CYBB;...,ccrcc,2.329713,1098.0,8,1203.125,1048.0
8185,Neutrophil degranulation,-0.234137,-1.460782,0.851782,0.853043,479,405,S100P;S100A9;S100A8;S100A11;S100A12;CAMP;LCN2;...,RNASET2;ITGAV;CEACAM1;ADAM10;CFP;CTSC;ACTR1B;M...,colon,1.460782,2009.0,8,1203.125,1048.0
8186,Neutrophil degranulation,0.382288,2.147134,0.365123,0.772177,479,427,S100A8;S100A9;FCER1G;LTF;S100A12;DSP;LCN2;CAMP...,S100A8;S100A9;FCER1G;LTF;S100A12;DSP;LCN2;CAMP...,endometrial,2.147134,1260.0,8,1203.125,1048.0
8187,Neutrophil degranulation,0.344896,1.817912,0.528384,0.817244,479,432,SERPINA1;HBB;S100A11;ANXA2;FCER1G;CD44;SERPINA...,SERPINA1;HBB;S100A11;ANXA2;FCER1G;CD44;SERPINA...,gbm,1.817912,1668.0,8,1203.125,1048.0
8188,Neutrophil degranulation,0.235027,1.492685,0.938557,1.000000,479,448,MGAM;AOC1;CEP290;CR1;FCER1G;CAMP;PLAU;EPX;MCEM...,MGAM;AOC1;CEP290;CR1;FCER1G;CAMP;PLAU;EPX;MCEM...,hnscc,1.492685,1368.0,8,1203.125,1048.0
8189,Neutrophil degranulation,-0.392260,-2.377807,0.338095,0.600050,479,423,DSP;PKP1;PLAU;SERPINB3;CKAP4;JUP;TNFAIP6;S100A...,RAB14;DOCK2;TMC6;CEACAM8;RAB4B;LRRC7;CRISPLD2;...,lscc,2.377807,1014.0,8,1203.125,1048.0
8190,Neutrophil degranulation,-0.432713,-2.553530,0.192096,0.480826,479,409,DSP;PLAU;TXNDC5;CKAP4;MLEC;NEU1;DNAJC3;FUCA1;H...,AZU1;S100A9;CEACAM8;STK10;SVIP;DEFA4;NAPRT;DNA...,luad,2.553530,773.0,8,1203.125,1048.0
8191,Neutrophil degranulation,-0.480385,-3.021381,0.045331,0.355939,479,418,NPC2;PKM;HSP90AB1;ILF2;MIF;SRP14;HSP90AA1;CD47...,PLEKHO2;HEBP2;ROCK1;CTSH;ARHGAP9;SDCBP;CR1;CPP...,ovarian,3.021381,435.0,8,1203.125,1048.0


In [30]:
find_neutrophil_degranulation(STEP5_log2_ratio_of_classes)

,Term,es,nes,pval,fdr,geneset_size,matched_size,genes,ledge_genes,cancer_type,abs_nes,cancer_rank_abs_nes,pathway_times_enriched,pathway_avg_rank,pathway_overall_rank
1772,Neutrophil degranulation,0.264286,1.039035,1.0,1.0,479,448,PTPRN2;P2RX1;AOC1;MMP25;NFASC;MCEMP1;MGAM;CEP2...,PTPRN2;P2RX1;AOC1;MMP25;NFASC;MCEMP1;MGAM;CEP2...,hnscc,1.039035,1972.0,1,1972.0,1773.0


In [31]:
find_neutrophil_degranulation(STEP5_ratio_of_classes)

,Term,es,nes,pval,fdr,geneset_size,matched_size,genes,ledge_genes,cancer_type,abs_nes,cancer_rank_abs_nes,pathway_times_enriched,pathway_avg_rank,pathway_overall_rank
11424,Neutrophil degranulation,-0.320748,-1.881164,0.699888,0.883099,479,427,PPIE;PSMD6;CHI3L1;RHOA;PSMD2;SERPINB10;DEFA4;R...,LRMP;PIGR;EEF2;VAT1;APRT;ATAD3B;RAC1;LYZ;FCGR3...,ccrcc,1.881164,1401.0,7,1498.142857,1745.0
11425,Neutrophil degranulation,-0.330949,-1.461890,0.913725,1.000000,479,427,TNFAIP6;CD300A;CD55;ALDH3B1;UNC13D;PGM1;LAMTOR...,FUCA1;ITGAV;SNAP29;LRMP;PYCARD;CEACAM8;MAN2B1;...,endometrial,1.461890,1352.0,7,1498.142857,1745.0
11426,Neutrophil degranulation,-0.486678,-1.629601,0.631119,1.000000,479,432,CYB5R3;CEACAM8;PLAUR;GM2A;ORMDL3;PSMB7;ATAD3B;...,BST2;SLC44A2;TLR2;QSOX1;LYZ;FGR;FUCA1;RAB6A;LP...,gbm,1.629601,1292.0,7,1498.142857,1745.0
11427,Neutrophil degranulation,0.134445,0.815357,0.588816,0.993212,479,448,PTPRN2;P2RX1;AOC1;MMP25;NFASC;MCEMP1;MGAM;CEP2...,PTPRN2;P2RX1;AOC1;MMP25;NFASC;MCEMP1;MGAM;CEP2...,hnscc,0.815357,2196.0,7,1498.142857,1745.0
11428,Neutrophil degranulation,0.286571,1.197760,0.874494,1.000000,479,423,MMP8;RETN;MAN2B1;GM2A;PADI2;FABP5;CD300A;KCMF1...,MMP8;RETN;MAN2B1;GM2A;PADI2;FABP5;CD300A;KCMF1...,lscc,1.197760,1941.0,7,1498.142857,1745.0
11429,Neutrophil degranulation,-0.395257,-2.082395,0.555435,1.000000,479,409,PSMB7;GHDC;RAB6A;HLA-B;TMC6;SNAP29;CHIT1;EPX;C...,HP;AGA;PSMD6;MME;HBB;FCAR;ITGAM;FCN1;XRCC6;P2R...,luad,2.082395,997.0,7,1498.142857,1745.0
11430,Neutrophil degranulation,0.418587,1.769275,0.391486,0.627585,479,418,DIAPH1;HSPA6;TUBB;GSDMD;LRMP;ALOX5;KRT1;ARSB;R...,DIAPH1;HSPA6;TUBB;GSDMD;LRMP;ALOX5;KRT1;ARSB;R...,ovarian,1.769275,1308.0,7,1498.142857,1745.0


In [32]:
find_neutrophil_degranulation(STEP5_signal_to_noise)

,Term,es,nes,pval,fdr,geneset_size,matched_size,genes,ledge_genes,cancer_type,abs_nes,cancer_rank_abs_nes,pathway_times_enriched,pathway_avg_rank,pathway_overall_rank
9672,Neutrophil degranulation,0.366073,2.955986,0.042945,0.256524,479,427,PYGL;TYROBP;FCER1G;ITGAX;RAP2B;LPCAT1;CYBB;ALD...,PYGL;TYROBP;FCER1G;ITGAX;RAP2B;LPCAT1;CYBB;ALD...,ccrcc,2.955986,437.0,8,1299.25,1248.0
9673,Neutrophil degranulation,0.177983,1.432615,0.814151,0.885181,479,405,S100A11;HSP90AB1;CNN2;XRCC6;S100P;ILF2;PA2G4;P...,S100A11;HSP90AB1;CNN2;XRCC6;S100P;ILF2;PA2G4;P...,colon,1.432615,1988.0,8,1299.25,1248.0
9674,Neutrophil degranulation,0.331414,2.435937,0.244565,0.659998,479,427,EEF2;COPB1;PSMD2;DYNLT1;DSP;JUP;PKM;NBEAL2;CTS...,EEF2;COPB1;PSMD2;DYNLT1;DSP;JUP;PKM;NBEAL2;CTS...,endometrial,2.435937,928.0,8,1299.25,1248.0
9675,Neutrophil degranulation,0.196746,1.279314,0.782756,0.921631,479,432,IDH1;PGM2;STK10;IQGAP2;EEF1A1;PYGL;SERPINB6;TY...,IDH1;PGM2;STK10;IQGAP2;EEF1A1;PYGL;SERPINB6;TY...,gbm,1.279314,2114.0,8,1299.25,1248.0
9676,Neutrophil degranulation,0.259328,2.089915,0.401060,0.557506,479,448,CNN2;PLAU;IGF2R;LPCAT1;FCER1G;HSP90AB1;ITGAX;M...,CNN2;PLAU;IGF2R;LPCAT1;FCER1G;HSP90AB1;ITGAX;M...,hnscc,2.089915,1687.0,8,1299.25,1248.0
9677,Neutrophil degranulation,-0.336553,-2.402527,0.306912,0.503667,479,423,EEF2;CKAP4;DDX3X;DSP;HSP90AB1;PLAU;PA2G4;ILF2;...,SIGLEC9;LRRC7;PTPRJ;OSTF1;PGM2;VNN1;HGSNAT;LIL...,lscc,2.402527,1145.0,8,1299.25,1248.0
9678,Neutrophil degranulation,-0.326192,-2.421644,0.250681,0.476034,479,409,COPB1;EEF2;SRP14;HSP90AB1;PSMD3;PSMD6;DNAJC3;P...,LILRB3;GSTP1;STK11IP;PAFAH1B2;COTL1;SLPI;MGAM;...,luad,2.421644,1109.0,8,1299.25,1248.0
9679,Neutrophil degranulation,-0.304473,-2.552527,0.224976,0.480426,479,418,HSP90AB1;HSP90AA1;ILF2;COPB1;EEF1A1;EEF2;NPC2;...,STK11IP;SLC15A4;HEBP2;SLPI;S100A9;FLG2;MNDA;TO...,ovarian,2.552527,986.0,8,1299.25,1248.0


In [33]:
find_neutrophil_degranulation(STEP5_t_test)

,Term,es,nes,pval,fdr,geneset_size,matched_size,genes,ledge_genes,cancer_type,abs_nes,cancer_rank_abs_nes,pathway_times_enriched,pathway_avg_rank,pathway_overall_rank
9728,Neutrophil degranulation,0.363073,2.930958,0.053556,0.258452,479,427,PYGL;TYROBP;FCER1G;ITGAX;RAP2B;CYBB;LPCAT1;ALD...,PYGL;TYROBP;FCER1G;ITGAX;RAP2B;CYBB;LPCAT1;ALD...,ccrcc,2.930958,472.0,8,1305.625,1257.0
9729,Neutrophil degranulation,0.172719,1.386814,0.846516,0.900781,479,405,S100A11;HSP90AB1;CNN2;XRCC6;S100P;ILF2;PA2G4;P...,S100A11;HSP90AB1;CNN2;XRCC6;S100P;ILF2;PA2G4;P...,colon,1.386814,2018.0,8,1305.625,1257.0
9730,Neutrophil degranulation,0.327291,2.395131,0.262635,0.693537,479,427,EEF2;COPB1;PSMD2;DYNLT1;DSP;JUP;PKM;PSMD1;NBEA...,EEF2;COPB1;PSMD2;DYNLT1;DSP;JUP;PKM;PSMD1;NBEA...,endometrial,2.395131,979.0,8,1305.625,1257.0
9731,Neutrophil degranulation,0.192165,1.253155,0.800509,0.925179,479,432,IDH1;PGM2;EEF1A1;IQGAP2;APAF1;STK10;CPNE3;TYRO...,IDH1;PGM2;EEF1A1;IQGAP2;APAF1;STK10;CPNE3;TYRO...,gbm,1.253155,2123.0,8,1305.625,1257.0
9732,Neutrophil degranulation,0.257080,2.070832,0.418358,0.568355,479,448,CNN2;PLAU;LPCAT1;IGF2R;FCER1G;HSP90AB1;MNDA;IT...,CNN2;PLAU;LPCAT1;IGF2R;FCER1G;HSP90AB1;MNDA;IT...,hnscc,2.070832,1702.0,8,1305.625,1257.0
9733,Neutrophil degranulation,-0.341842,-2.443492,0.292976,0.477470,479,423,EEF2;CKAP4;DDX3X;DSP;HSP90AB1;ILF2;PLAU;PA2G4;...,C5AR1;CTSZ;FTL;GAA;CAPN1;HK3;BST1;RAB31;COTL1;...,lscc,2.443492,1113.0,8,1305.625,1257.0
9734,Neutrophil degranulation,-0.331756,-2.459892,0.233908,0.459798,479,409,COPB1;SRP14;EEF2;PSMD3;HSP90AB1;PSMD6;PDAP1;DN...,CAMP;STK11IP;CTSD;COTL1;LTF;MGAM;DOCK2;SLPI;IQ...,luad,2.459892,1060.0,8,1305.625,1257.0
9735,Neutrophil degranulation,-0.305249,-2.558660,0.225191,0.476000,479,418,HSP90AB1;HSP90AA1;ILF2;COPB1;EEF1A1;EEF2;PKM;X...,STK11IP;AP2A2;SLC15A4;TOM1;HEBP2;SLPI;S100A9;H...,ovarian,2.558660,978.0,8,1305.625,1257.0


## Figure 2: Expression of proteins in the neutrophil degranulation pathway

In [ ]:
# Get the ID the top expessed pathway
neutro_pathway_ids = enrichment_data.\
    loc[enrichment_data["name"] == "Neutrophil degranulation", "pathway_id"].\
    unique()

if len(neutro_pathway_ids) != 1:
    raise ValueError("Multiple pathways found.")
    
neutro_pathway_id = neutro_pathway_ids[0]

### A: With proteins in different orders for the up and down categories

In [ ]:
def expr_heatmap(expr_data, enrich_data, pathway_id, up_or_down, max_p, x_title=True):
    """Make a heatmap with proteins on the X axis, cancer types on the Y 
    axis, and the color reflecting the expression of that protein in that
    cancer type.
    
    Parameters:
    expr_data (pandas.DataFrame): A dataframe with columns "cancer_type", "protein_str", "adj_p", and "simplified_change".
    enrich_data (pandas.DataFrame): A dataframe with columns "cancer_type" and "mean_expr" (i.e. mean expression for the pathway of interest).
    pathway_id (str): The Reactome pathway id for the pathway you want the heatmap for.
    up_or_down (str): Either "up" or "down"; whether to plot cancers where the pathway is up, or where it's down.
    max_p (float): The minimum p value for a protein to be included on the plot. If max_p > 0.05, then proteins with 0.05 <= p < max_p will be marked as marginally significant.
    x_title (bool): Whether to include an x axis title.
    
    Returns:
    altair.Chart: The heatmap.
    """
    # Filter expression data by p value
    expr_data = expr_data[expr_data["adj_p"] < max_p]
    
    # Get expression data for proteins in the pathway
    pathway_proteins = ut.search_reactome_proteins_in_pathways(pathway_id)
    pathway_expr = expr_data[expr_data["protein_str"].isin(pathway_proteins["member"])]
    
    # Categorize the cancer types as the neutrophil degranulation pathway's mean expression being up or down
    if up_or_down == "up":
        comparison = operator.gt
    elif up_or_down == "down":
        comparison = operator.lt
    else:
        raise ValueError(f"Invalid value for up_or_down parameter: '{up_or_down}'")
    
    sel_cancers = enrich_data.\
        loc[(enrich_data["pathway_id"] == neutro_pathway_id) & (comparison(enrich_data["mean_expr"], 0)),
           ["cancer_type", "mean_expr"]]

    # Select data for the specified cancer types
    selected_expr_data = pathway_expr[pathway_expr["cancer_type"].isin(sel_cancers["cancer_type"])]

    # Sort proteins by mean expression across selected cancer types
    prot_order = selected_expr_data.groupby("protein_str")[["simplified_change", "cancer_type"]].\
        agg({
            "simplified_change": np.mean,
            "cancer_type": "count"
        })

    prot_order = prot_order.assign(
            cancer_type=np.copysign(prot_order["cancer_type"], prot_order["simplified_change"])
        ).\
        sort_values(by=["simplified_change", "cancer_type"])

    # Sort cancer types by mean pathway expression
    cancer_order = sel_cancers.\
        sort_values(by="mean_expr", ascending=False).\
        loc[:, "cancer_type"]
    
    # Map simplified_change to legend values
    selected_expr_data = selected_expr_data.assign(
        tumor_up_down=selected_expr_data["simplified_change"].replace({
            1: "Up (adj p < 0.05)",
            MSV: f"Up (0.05 <= adj p < {max_p})",
            0: "No change",
            -MSV: f"Down (0.05 <= adj p < {max_p})",
            -1: "Down (adj p < 0.05)"
        })
    )
    
    # Plot
    plot = alt.Chart(selected_expr_data).mark_rect().encode(
        x=alt.X(
            "protein_str:N",
            sort=prot_order.index.values,
            axis=alt.Axis(
                labels=False,
                ticks=False,
                title=f"Proteins for {pathway_id} (different order in each facet)" if x_title else ""
            )
        ),
        y=alt.Y(
            "cancer_type:N",
            sort=cancer_order.values,
            axis=alt.Axis(
                title=f"Pathway {up_or_down} in tumor"
            )
        ),
        color=alt.Color(
            "tumor_up_down:N",
            scale=alt.Scale(
                domain=[
                    "Up (adj p < 0.05)",
                    f"Up (0.05 <= adj p < {max_p})",
                    "No change",
                    f"Down (0.05 <= adj p < {max_p})",
                    "Down (adj p < 0.05)"
                ],
                range=["#bf363a",
                       "#f6bda4",
                       "#f2efee",
                       "#afd3e6",
                       "#1e588a"
                ]
            ),
            legend=alt.Legend(
                title="Expression in tumor"
            )
        ),
    ).properties(
        width=700,
        height=175
    )
    
    return plot

In [ ]:
# Plot
alt.vconcat(
    expr_heatmap(all_expression_data, enrichment_data, neutro_pathway_id, "up", max_p=0.2, x_title=False),
    expr_heatmap(all_expression_data, enrichment_data, neutro_pathway_id, "down", max_p=0.2)
)

### B: With same sorting for both

In [ ]:
def expr_heatmap_alt_sort(expr_data, enrich_data, pathway_id, max_p):
    """Make a heatmap with proteins on the X axis, cancer types on the Y 
    axis, and the color reflecting the expression of that protein in that
    cancer type.
    
    Parameters:
    expr_data (pandas.DataFrame): A dataframe with columns "cancer_type", "protein_str", "adj_p", and "simplified_change".
    enrich_data (pandas.DataFrame): A dataframe with columns "cancer_type" and "mean_expr" (i.e. mean expression for the pathway of interest).
    pathway_id (str): The Reactome pathway id for the pathway you want the heatmap for.
    up_or_down (str): Either "up" or "down"; whether to plot cancers where the pathway is up, or where it's down.
    max_p (float): The minimum p value for a protein to be included on the plot. If max_p > 0.05, then proteins with 0.05 <= p < max_p will be marked as marginally significant.
    
    Returns:
    altair.Chart: The heatmap.
    """
    # Filter expression data by p value
    expr_data = expr_data[expr_data["adj_p"] < max_p]
    
    # Get expression data for proteins in the pathway
    pathway_proteins = ut.search_reactome_proteins_in_pathways(pathway_id)
    pathway_expr = expr_data[expr_data["protein_str"].isin(pathway_proteins["member"])]
    
    # Sort proteins by mean expression across all cancer types
    prot_order = pathway_expr.groupby("protein_str")[["simplified_change", "cancer_type"]].\
        agg({
            "simplified_change": np.mean,
            "cancer_type": "count"
        })

    prot_order = prot_order.assign(
            cancer_type=np.copysign(prot_order["cancer_type"], prot_order["simplified_change"])
        ).\
        sort_values(by=["simplified_change", "cancer_type"])

    # Sort cancer types by mean pathway expression
    sel_cancers = enrich_data.\
        loc[enrich_data["pathway_id"] == neutro_pathway_id, ["cancer_type", "mean_expr"]]
        
    sel_cancers = sel_cancers.\
        assign(up_or_down=np.where(sel_cancers["mean_expr"] > 0, "Pathway up in tumor", "Pathway down in tumor")).\
        sort_values(by="mean_expr", ascending=False).\
        loc[:, ["cancer_type", "up_or_down"]]
    
    cancer_order = sel_cancers["cancer_type"].values
    
    pathway_expr = pathway_expr.merge(
        right=sel_cancers,
        how="inner",
        left_on="cancer_type",
        right_on="cancer_type",
        validate="many_to_one"
    )
    
    # Map simplified_change to legend values
    pathway_expr = pathway_expr.assign(
        tumor_up_down=pathway_expr["simplified_change"].replace({
            1: "Up (adj p < 0.05)",
            MSV: f"Up (0.05 <= adj p < {max_p})",
            0: "No change",
            -MSV: f"Down (0.05 <= adj p < {max_p})",
            -1: "Down (adj p < 0.05)"
        })
    )
    
    # Plot
    plot = alt.Chart(pathway_expr).mark_rect().encode(
        x=alt.X(
            "protein_str:N",
            sort=prot_order.index.values,
            axis=alt.Axis(
                labels=False,
                ticks=False,
                title=f"Proteins for {pathway_id} (same order in both facets)"
            )
        ),
        y=alt.Y(
            "cancer_type:N",
            sort=cancer_order,
            axis=alt.Axis(
                title=""
            )
        ),
        color=alt.Color(
            "tumor_up_down:N",
            scale=alt.Scale(
                domain=[
                    "Up (adj p < 0.05)",
                    f"Up (0.05 <= adj p < {max_p})",
                    "No change",
                    f"Down (0.05 <= adj p < {max_p})",
                    "Down (adj p < 0.05)"
                ],
                range=["#bf363a",
                       "#f6bda4",
                       "#f2efee",
                       "#afd3e6",
                       "#1e588a"
                ]
            ),
            legend=alt.Legend(
                title="Expression in tumor"
            )
        ),
        row=alt.Row(
            "up_or_down:N",
            sort="descending",
            title="",
            header=alt.Header(
                labelFontStyle="bold"
            )
        )
    ).properties(
        width=600,
        height=150
    ).resolve_scale(
        y="independent" # This makes it so each facet (up or down) only has the cancers that are in that category
    )
    
    return plot

In [ ]:
# Plot
expr_heatmap_alt_sort(all_expression_data, enrichment_data, neutro_pathway_id, max_p=0.2)

## Figure 3: Pathway overlay for neutrophil degranulation

In [ ]:
def pathway_overlay_wrapper(pathway_id, exp_data, enrich_data, up_or_down):
    
    prots = ut.search_reactome_proteins_in_pathways(pathway_id)
    
    # Categorize the cancer types as the neutrophil degranulation pathway's mean expression being up or down
    if up_or_down == "up":
        comparison = operator.gt
    elif up_or_down == "down":
        comparison = operator.lt
    else:
        raise ValueError(f"Invalid value for up_or_down parameter: '{up_or_down}'")
    
    sel_cancers = enrich_data.\
        loc[(enrich_data["pathway_id"] == neutro_pathway_id) & (comparison(enrich_data["mean_expr"], 0)),
            "cancer_type"]
    
    # Select the desired expression data and average the simplified change across cancer types
    sel_exp = exp_data.\
        loc[
            exp_data["protein_str"].isin(prots["member"]) & 
            exp_data["cancer_type"].isin(sel_cancers),
            ["protein_str", "simplified_change"]
        ].\
        groupby("protein_str").\
        agg(np.mean).\
        rename(columns={"simplified_change": f"{up_or_down}_simp_change"})
    
    img_name = f"{TIME_START}_{pathway_id}_{up_or_down}.png"
    img_path = os.path.join(STEP7_DIR, img_name)
    
    token, _ = ut.reactome_enrichment_analysis(
        analysis_type="ranked", 
        data=sel_exp, 
        sort_by="ENTITIES_FDR", 
        ascending=True,
        include_high_level_diagrams=True, 
        include_interactors=False)
    
    _, img_path = ut.reactome_pathway_overlay(
        analysis_token=token,
        pathway=pathway_id,
        open_browser=False,
        export_path=img_path,
        image_format="png",
        display_col_idx=None,
        diagram_colors="Modern",
        overlay_colors="Standard",
        quality=10
    )

    _, url = ut.reactome_pathway_overlay(
        analysis_token=token,
        pathway=pathway_id,
        open_browser=False,
    )

    return img_path, url

In [ ]:
# Note that we're using expression_data, not all_expression_data; the former has
# only proteins where reject_null

up_image_path, up_url = pathway_overlay_wrapper(neutro_pathway_id, expression_data, enrichment_data, "up")

In [ ]:
IPython.display.HTML(f'<a href={up_url}>{up_url}</a>')

In [ ]:
IPython.display.Image(up_image_path)

In [ ]:
# Note that we're using expression_data, not all_expression_data; the former has
# only proteins where reject_null

down_image_path, down_url = pathway_overlay_wrapper(neutro_pathway_id, expression_data, enrichment_data, "down")

In [ ]:
IPython.display.HTML(f'<a href={down_url}>{down_url}</a>')

In [ ]:
IPython.display.Image(down_image_path)